### Bringing in document group data to make reprint group id metadata
Followed by merging reprint document groups with large metadata sheet

In [1]:
import pandas as pd

In [2]:

# ---------- CONFIG ----------
WIDE_PATH = "document_groups.csv"
LABELED_OUT = "document_groups_labeled.csv"

# 1. Load the wide table
df = pd.read_csv(WIDE_PATH)

# Drop any stray unnamed columns from Excel
df = df.loc[:, ["article_id_parent", "article_id_direct",
                "article_id_truncated", "article_id_paraphrase"]]

# 2. Normalize blanks to NaN
for col in df.columns:
    df[col] = df[col].replace(r"^\s*$", pd.NA, regex=True)

# 3. Forward-fill parent so every row knows its group parent
df["group_parent"] = df["article_id_parent"].ffill()

# Drop rows with no parent at all (just in case)
df = df[df["group_parent"].notna()].copy()

# Base group id type: parent + "_reprint"
df["group_id_type"] = df["group_parent"].astype(str) + "_reprint"

# 4. Build long-form pieces

# a) Originals
orig = (
    df.loc[df["article_id_parent"].notna(), ["article_id_parent", "group_id_type"]]
      .rename(columns={"article_id_parent": "article_id"})
)
orig["reprint_type"] = "original"

# b) Direct reprints
direct = (
    df.loc[df["article_id_direct"].notna(), ["article_id_direct", "group_id_type"]]
      .rename(columns={"article_id_direct": "article_id"})
)
direct["reprint_type"] = "direct"

# c) Truncated reprints
trunc = (
    df.loc[df["article_id_truncated"].notna(), ["article_id_truncated", "group_id_type"]]
      .rename(columns={"article_id_truncated": "article_id"})
)
trunc["reprint_type"] = "truncated"

# d) Paraphrase reprints (NEW)
para = (
    df.loc[df["article_id_paraphrase"].notna(), ["article_id_paraphrase", "group_id_type"]]
      .rename(columns={"article_id_paraphrase": "article_id"})
)
para["reprint_type"] = "paraphrase"

# 5. Combine them
out = pd.concat([orig, direct, trunc, para], ignore_index=True)

# Drop duplicates just in case the same article_id appears more than once
out = out.drop_duplicates(subset=["article_id", "group_id_type", "reprint_type"])

# 6. Save
out.to_csv(LABELED_OUT, index=False)
print(f"Saved labeled reprints (with paraphrase) to: {LABELED_OUT}")


Saved labeled reprints (with paraphrase) to: document_groups_labeled.csv


In [3]:
import pandas as pd

# ---------- CONFIG ----------
META_PATH = "complete_metadata_images_tropes_reprints_transcripts.csv"  # your current master
REPRINTS_PATH = "document_groups_labeled.csv"
OUTPUT_PATH = "complete_metadata_images_tropes_reprints_transcripts.csv"

# 1. Load both
meta = pd.read_csv(META_PATH)
reprints = pd.read_csv(REPRINTS_PATH)

# 2. Normalize article_id
meta["article_id"] = meta["article_id"].astype(str).str.strip()
reprints["article_id"] = reprints["article_id"].astype(str).str.strip()

# 3. Rename group_id_type to group_reprint_id for clarity
reprints = reprints.rename(columns={"group_id_type": "group_reprint_id"})

# 4. Drop any existing reprint columns in meta (so we overwrite with the new full set)
meta = meta.drop(columns=["group_reprint_id", "reprint_type"], errors="ignore")

# 5. Merge on article_id (LEFT JOIN: keep all metadata rows)
meta_merged = meta.merge(
    reprints[["article_id", "group_reprint_id", "reprint_type"]],
    on="article_id",
    how="left"
)

# 6. Optional: replace NaN with empty strings
meta_merged["group_reprint_id"] = meta_merged["group_reprint_id"].fillna("")
meta_merged["reprint_type"] = meta_merged["reprint_type"].fillna("")

# 7. Save
meta_merged.to_csv(OUTPUT_PATH, index=False)
print(f"Saved updated master metadata to: {OUTPUT_PATH}")


Saved updated master metadata to: complete_metadata_images_tropes_reprints_transcripts.csv


In [5]:
# sanity check

meta_merged[meta_merged["article_id"] == "SanFranciscoDailyHerald1853"][
    ["title", "publication", "object_id", "article_id", "group_reprint_id", "reprint_type"]
]


,title,publication,object_id,article_id,group_reprint_id,reprint_type
2624,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_Original,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
2625,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_img1,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
2626,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_img2,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
2627,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_img3,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
2628,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_img4,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
2629,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_img5,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
2630,"""The Wild Woman of San Nicolas""",San Francisco Daily Herald,SanFranciscoDailyHerald1853_img6,SanFranciscoDailyHerald1853,Santiago1853_reprint,paraphrase
